In [ ]:
import time
import yaml
import os
import threading
import time
from datetime import datetime
import cv2
import numpy as np
import torch
import tensorflow as tf

In [ ]:
from AccidentError import AccidentError
from NavigationBase import NavigationBase
from lane_detection.main import get_lanes
from distance_base_navigation import run_navigation as distance_nav_loop
from autonomous_navigation import run_navigation as auto_nav_loop

In [3]:
import easygopigo3 as easy

my_gpg3 = easy.EasyGoPiGo3(use_mutex=True)

ModuleNotFoundError: No module named 'easygopigo3'

In [1]:
from ultralytics import YOLO
model = YOLO("stop_sign_detection\\weights\\best.pt")
model.export(format="tflite") 

Ultralytics 8.4.38  Python-3.11.9 torch-2.8.0+cu129 CPU (11th Gen Intel Core i7-11800H @ 2.30GHz)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from 'stop_sign_detection\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.3 MB)
requirements: Ultralytics requirements ['tf_keras<=2.19.0', 'sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx2tf>=1.26.3,<1.29.0', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------------ --------- 1.3/1.7 MB 7.5 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 7.2 MB/s  0:00:00
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/12.8 MB 7.0 MB/s eta 0:00:02
   --------- ---------

c:\Users\vicki\Documents\ECSU_Courses\Fall 2025 COURSE FILES\Big Data Independent Study\run-model\Lib\site-packages\torch\onnx\utils.py:1397: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.72...
ONNX: export success  2.6s, saved as 'stop_sign_detection\weights\best.onnx' (10.2 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to C:\Users\vicki\Documents\ECSU_Courses\Spring 2026 COURSE FILES\Senior Research\Autonomous-Robot-Navigation-System\calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 79.8files/s 0.0s

TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at 'stop_sign_detection\weights\best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 5, 8400), dtype=tf.float32, name=None)
Captures:
  2126745421584: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  2126742397264: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  2126745425808: TensorSpec(shape=(16,), dtype=tf.float

'stop_sign_detection\\weights\\best_saved_model\\best_float32.tflite'

In [ ]:
# collect instructions
def get_instructions_from_file(filename):
    drive_path = {"forward": [], "turn": []}
    path_order = []
    with open(filename, 'r') as file:
        drive_path = {"forward": [], "turn": []}
        path_order = []
        data = yaml.safe_load(file)
        steps = data["steps"]
        for i,step in enumerate(steps):
            drive_path[list(step.keys())[0]].append(list(step.values())[0])
            path_order.append(list(step.keys())[0])
    return drive_path, path_order

In [ ]:
def driving_loop(nav_base,direction,dist_or_angle):
    if nav_error.is_set(): return
    nav_base.robot.reset_encoders(blocking=False)
    if direction == "forward":
        nav_base.forward(dist_or_angle)
        #print("Estimated length of drive section (seconds)",((dist_or_angle/nav_base.wheel_circumference)*360) / nav_base.robot.get_speed())
        target = (dist_or_angle / nav_base.wheel_circumference) * 360
        while not nav_base.robot.target_reached(target, target):
            time.sleep(0.1)
        #time.sleep(((dist_or_angle/nav_base.wheel_circumference)*360) / nav_base.robot.get_speed())
    elif direction == "turn":
        nav_base.turn(dist_or_angle)
        #print("Estimated length of turn (seconds)",((((abs(dist_or_angle)/90)*(2*math.pi*turn_radius))/nav_base.wheel_circumference)*360) / nav_base.robot.get_speed())
        target = (((abs(dist_or_angle)/360)*(2*math.pi*nav_base.turn_radius))/nav_base.wheel_circumference) * 360
        while not nav_base.robot.target_reached(target, target):
            time.sleep(0.1)
        #time.sleep(((((abs(dist_or_angle)/90)*(2*math.pi*nav_base.turn_radius))/nav_base.wheel_circumference)*360) / nav_base.robot.get_speed())
        nav_base.robot.reset_speed()


In [ ]:
def follow_instructions(path, order, nav_base):
    distance_traveled = 0
    for direction in order:
        if nav_error.is_set():
            break
        dist_or_angle = path[direction].pop(0)
        if direction=="forward": distance_traveled += dist_or_angle
        elif direction=="turn": distance_traveled += (abs(dist_or_angle)/360)*(2*math.pi*turn_radius)
        driving_loop(nav_base,direction=direction,dist_or_angle=dist_or_angle)
    with open(os.path.join(save_dir,f"trial_{trial_num}_distance_traveled.yml"),"w") as file:
        yaml.dump({"distance_traveled":distance_traveled},file)

In [ ]:
def run_camera(nav_base):
    if nav_base.camera is not None:
        ret,frame = nav_base.camera.read()
        if ret: 
            interpreter = tf.lite.Interpreter(model_path=os.path.join("stop_sign_detection","weights","best_saved_model","best_float32.tflite"))
            interpreter.allocate_tensors()
            total_time = 0.0
            frames_processed = 0
            while nav_base.camera.isOpened():
                ret,frame = nav_base.camera.read()
                if ret:
                    frames_processed += 1
                    start_time = time.time()
                    lane_thread = threading.Thread(target=stay_in_lane,args=(frame,nav_base))
                    lane_thread.start()
                    stop_thread = threading.Thread(target=obey_stop,args=(frame,interpreter,nav_base))
                    stop_thread.start()
                    lane_thread.join()
                    stop_thread.join()
                    total_time += time.time() - start_time
                else:
                    print("Error: Could not read the frame.")
                    break
                if nav_error.is_set():
                    break
            with open(os.path.join(save_dir,f"trial_{trial_num}_average_fps.yml"),"w") as file:
                yaml.dump({"fps":total_time/frames_processed, "frames":frames_processed},file)
    else: print ("Camera is not initialized")

In [ ]:
def stay_in_lane(img,nav_base):
    if not ignore_lane.is_set():
        print(get_lanes(img))
        # run lane model
        # check wheel to line difference
        # if too far right, shift left
        # if too far left, shift right
        # ignore if set
        pass

In [ ]:
def obey_stop(img,interpreter,nav_base):
    if not stop_sign.is_set():
        resize = cv2.resize(img, (640,640))
        
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()

        interpreter.set_tensor(input_details[0]['index'], resize)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details[0]['index'])
        print(output_data)
    # check for stop sign
    # if sign:
    #   check distance to stop sign
    #   slow down to slowest speed, then total stop
    # navigation tells when to go again

In [ ]:
def emergency_stop(nav_base):
    while True:
        nav_base.emergency_stop()
        time.sleep(0.005)

In [ ]:
def select_nav_system(nav_type):
    if nav_type is None:
        return None
    if nav_type == "distance":
        return distance_nav_loop
    if nav_base == "autonomous":
        return auto_nav_loop
    else:
        return None

In [ ]:
# Testing drive paths
def main():
    #instruction_dir = "drive_instructions"
    #instruction_files = list(map(lambda file: os.path.join(instruction_dir, file),os.listdir(instruction_dir)))
    global nav_sys, camera_ip, trials,trial_num,drive_file,save_dir
    with open("trial_config.yml","r+") as config_file:
        config_dict = yaml.safe_load(config_file)
        if config_dict["n_trials"] > config_dict["last_trial"]: 
            config_dict["last_trial"] += 1
            yaml.dump(config_dict,config_file)
            nav_type,camera_ip,trials,trial_num,drive_file,save_dir = config_dict.values()
            nav_sys = select_nav_system(nav_type)

    if trial_num is not None:
        path,order = get_instructions_from_file(filename=drive_file)

        nav_base = NavigationBase(my_gpg3,camera_ip)
        
        global nav_error, ignore_lane, stop_sign
        nav_error = threading.Event()
        ignore_lane = threading.Event()
        stop_sign = threading.Event()
        
        try:
            emergency_stop_thread = threading.Thread(target=emergency_stop,args=(nav_base,),daemon=True)
            emergency_stop_thread.start()
            drive_thread = threading.Thread(follow_instructions,args=(path,order,nav_base,),daemon=True)
            drive_thread.start()
            camera_thread = threading.Thread(target=run_camera,args=(nav_base,),daemon=True)
            camera_thread.start()
            if nav_sys is not None:
                nav_thread = threading.Thread(target=nav_sys,args=(nav_base,),daemon=True)
                nav_thread.start()
            emergency_stop_thread.join()
        except AccidentError:
            print("Accident occured")
            nav_error.set()
        except:
            print("Unknown error")
    else:
        print("All trials completed")


In [ ]:
# get instructions
# run navigation
#   follow instructions -> forward or turn
#   while following instructions, should stay within lane and obey stop signs
#   while following instructions, should check surrounding & do actions based on surrounding
#       ex. check with distance, pathfinding, and potentially nothing
#   benchmark navigation - manage metric collection

In [ ]:
main()

In [ ]:
# Example initiation
nav_base = NavigationBase(my_gpg3)

In [ ]:
my_gpg3.forward()
try:
    while True:
        nav_base.emergency_stop()
        time.sleep(0.005)
except:
    print("Accident occured")